In [1]:
import numpy as np
np.random.seed(42)

In [2]:
import pandas as pd

crime_df = pd.read_csv("../data/processed/crime_base.csv")
college_df = pd.read_csv("../data/processed/college_base.csv")


crime_df.head()
college_df.head()

,County,Annual Number of Public High School Graduates,Public School Student-Teacher Ratio,Public School Expenditures Per Pupil ($ Dollars),Bachelor's Degree Attainment (%),High School Attainment (%),Number of 2-Year Colleges,Number of 4-Year Colleges and Universities,2-Year College Enrollment,4-Year College/University Enrollment
0,Allegany County,642,13.9,13771,18.3,90,1,1,2586,4831
1,Anne Arundel County,5268,14.7,13648,40.9,92.1,1,2,12655,548
2,Baltimore City,4108,16.1,15376,31.2,84.9,1,11,4909,59254
3,Baltimore County,7171,15.1,13880,38.6,91.1,1,7,17732,42571
4,Calvert County,1227,15.7,14062,31.1,93.9,0,0,NaN,NaN


In [3]:
crime_df.columns
college_df.columns

Index(['County', 'Annual Number of Public High School Graduates',
       'Public School Student-Teacher Ratio',
       'Public School Expenditures Per Pupil ($ Dollars)',
       'Bachelor's Degree Attainment (%)', 'High School Attainment (%)',
       'Number of 2-Year Colleges',
       'Number of 4-Year Colleges and Universities',
       '2-Year College Enrollment', '4-Year College/University Enrollment'],
      dtype='object')

In [4]:
crime_df["county_clean"] = crime_df["JURISDICTION"].str.lower().str.strip()
college_df["county_clean"] = college_df["County"].str.lower().str.strip()

In [5]:
print(list(crime_df.columns))
print(list(college_df.columns))

['JURISDICTION', 'YEAR', 'POPULATION', 'MURDER', 'RAPE', 'ROBBERY', 'AGG. ASSAULT', 'B & E', 'LARCENY THEFT', 'M/V THEFT', 'GRAND TOTAL', 'PERCENT CHANGE', 'VIOLENT CRIME TOTAL', 'VIOLENT CRIME PERCENT', 'VIOLENT CRIME PERCENT CHANGE', 'PROPERTY CRIME TOTALS', 'PROPERTY CRIME PERCENT', 'PROPERTY CRIME PERCENT CHANGE', 'OVERALL CRIME RATE PER 100,000 PEOPLE', 'OVERALL PERCENT CHANGE PER 100,000 PEOPLE', 'VIOLENT CRIME RATE PER 100,000 PEOPLE', 'VIOLENT CRIME RATE PERCENT CHANGE PER 100,000 PEOPLE', 'PROPERTY CRIME RATE PER 100,000 PEOPLE', 'PROPERTY CRIME RATE PERCENT CHANGE PER 100,000 PEOPLE', 'MURDER PER 100,000 PEOPLE', 'RAPE PER 100,000 PEOPLE', 'ROBBERY PER 100,000 PEOPLE', 'AGG. ASSAULT PER 100,000 PEOPLE', 'B & E PER 100,000 PEOPLE', 'LARCENY THEFT PER 100,000 PEOPLE', 'M/V THEFT PER 100,000 PEOPLE', 'MURDER  RATE PERCENT CHANGE PER 100,000 PEOPLE', 'RAPE RATE PERCENT CHANGE PER 100,000 PEOPLE', 'ROBBERY RATE PERCENT CHANGE PER 100,000 PEOPLE', 'AGG. ASSAULT  RATE PERCENT CHANGE

In [6]:
#college dataset


# Make numeric 
college_df["2-Year College Enrollment"] = pd.to_numeric(
    college_df["2-Year College Enrollment"], errors="coerce"
)
college_df["4-Year College/University Enrollment"] = pd.to_numeric(
    college_df["4-Year College/University Enrollment"], errors="coerce"
)


college_df_clean = college_df[[
    "county_clean",
    "Annual Number of Public High School Graduates",
    "2-Year College Enrollment",
    "4-Year College/University Enrollment"
]].copy()

#total college enrollment

college_df_clean["college_enrollment_total"] = (
    college_df_clean["2-Year College Enrollment"].fillna(0)
    + college_df_clean["4-Year College/University Enrollment"].fillna(0)
)

#renaming to simpler terms

college_df_clean = college_df_clean.rename(columns={
    "2-Year College Enrollement": "college_2yr",
    "4-Year College/University Enrollment": "enroll_4yr"
})

In [7]:
#Crime dataset

crime_df_clean = crime_df[[
    "county_clean",
    "GRAND TOTAL",
    "POPULATION",
    "YEAR",
   "VIOLENT CRIME TOTAL",
   "VIOLENT CRIME RATE PER 100,000 PEOPLE"
]].copy()

#renaming
crime_df_clean = crime_df_clean.rename(columns={
    "Violent Crime": "violent_crime"
})

In [8]:
crime_df["VIOLENT CRIME TOTAL"] = pd.to_numeric(
    crime_df["VIOLENT CRIME TOTAL"], errors="coerce"
)

crime_df["VIOLENT CRIME RATE PER 100,000 PEOPLE"] = pd.to_numeric(
    crime_df["VIOLENT CRIME RATE PER 100,000 PEOPLE"], errors="coerce"
)

In [9]:
#drop invalid rows
crime_df_clean = crime_df.dropna(subset=[
    "county_clean",
    "VIOLENT CRIME TOTAL",
    "VIOLENT CRIME RATE PER 100,000 PEOPLE"
])

college_df_clean = college_df_clean.dropna(subset=["county_clean"])

In [10]:
crime_df_clean.shape
college_df_clean.shape

(28, 5)

In [11]:
#select crime columns
crime_df_final = crime_df_clean[[
    "county_clean",
    "YEAR",
    "VIOLENT CRIME TOTAL",
    "VIOLENT CRIME RATE PER 100,000 PEOPLE",
    "POPULATION"
]].copy()

#renaming
crime_df_final = crime_df_final.rename(columns={
    "YEAR": "year",
    "VIOLENT CRIME TOTAL": "crime_total",
    "VIOLENT CRIME RATE PER 100,000 PEOPLE": "crime_rate",
    "POPULATION": "population"
 })

 #aggregate crime by county
crime_agg = crime_df_final.groupby("county_clean").agg({
    "crime_total": "mean",
    "crime_rate": "mean",
    "population": "mean"
 }).reset_index()

In [12]:
#merge
merged_df = pd.merge(
    crime_agg,
    college_df_clean,
    on = "county_clean",
    how = "inner"
)

merged_df.shape
merged_df.head()

,county_clean,crime_total,crime_rate,population,Annual Number of Public High School Graduates,2-Year College Enrollment,enroll_4yr,college_enrollment_total
0,allegany county,231.804348,309.823913,75763.413043,642,2586.0,4831.0,7417.0
1,anne arundel county,2185.021739,462.930435,466645.108696,5268,12655.0,548.0,13203.0
2,baltimore city,14311.978261,2021.226087,703032.391304,4108,4909.0,59254.0,64163.0
3,baltimore county,5659.152174,774.989130,738704.956522,7171,17732.0,42571.0,60303.0
4,calvert county,187.608696,311.117391,64829.152174,1227,NaN,NaN,0.0


In [13]:
merged_df.to_csv("../data/processed/MD_merged_crime_college.csv", index=False)

In [14]:
merged_df.describe()

,crime_total,crime_rate,population,2-Year College Enrollment,enroll_4yr,college_enrollment_total
count,24.000000,24.000000,24.000000,16.000000,13.000000,24.000000
mean,1507.889493,486.201721,213571.129529,7093.437500,18452.384615,14724.000000
std,3188.159950,369.883653,268353.077955,5957.557564,32142.833570,27882.582798
min,58.413043,210.139130,18595.782609,648.000000,548.000000,0.000000
25%,168.195652,310.794022,36389.744565,2814.750000,1512.000000,1128.000000
50%,337.641304,367.691304,83224.804348,5307.000000,3085.000000,4770.500000
75%,633.380435,513.292935,206802.032609,9952.250000,8617.000000,10966.500000
max,14311.978261,2021.226087,820118.086957,21260.000000,106543.000000,118332.000000


In [15]:
merged_df["county_clean"].nunique(), len(merged_df)

(24, 24)

In [16]:
merged_df[merged_df["county_clean"].duplicated(keep=False)]

,county_clean,crime_total,crime_rate,population,Annual Number of Public High School Graduates,2-Year College Enrollment,enroll_4yr,college_enrollment_total


In [17]:
merged_df[["crime_rate", "college_enrollment_total"]].corr()

,crime_rate,college_enrollment_total
crime_rate,1.000000,0.604268
college_enrollment_total,0.604268,1.000000
